# Bid list filter


## Original with tkinter

In [ ]:
# import pandas as pd
# from tkinter import Tk, filedialog
# from IPython.display import display

# root = Tk()
# root.withdraw()  # Hide the empty Tkinter window

# file_path = filedialog.askopenfilename(
#     title="Attach Bid List",
#     filetypes=[("Excel files", "*.xlsx")],
# )

# root.destroy()

# if file_path:
#     df = pd.read_excel(file_path)
#     display(df)
# else:
#     print("No file selected.")

## With ipywidgets

In [ ]:
import pandas as pd
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the upload button and space below it for messages
upload = widgets.FileUpload(accept='.xlsx', multiple=False)
output = widgets.Output()

# Function to display success when upload
def confirm_upload(change):
    with output:
        clear_output()
        if not upload.value:
            return

        # ipywidgets 8 uses a tuple; ipywidgets 7 uses a dictionary
        if isinstance(upload.value, dict):
            file_name = next(iter(upload.value))
        else:
            file_name = upload.value[0]['name']

        print(f"{file_name} attached.")

# Make upload button see the function
upload.observe(confirm_upload, names='value')

# Display button and space below 
display(upload, output)

In [ ]:
# Extracts data from widget and make it a df
if upload.value:
    content = upload.value[0]['content']
    file_name = upload.value[0]['name']
    df = pd.read_excel(io.BytesIO(content))
    display(df)
else:
    print("No file attached yet.")


df

## More Robust Set up with Validations
Outputs eligible and ineligible coils

In [36]:
import io
import re

import pandas as pd


MAX_WIDTH_INCHES = 62
MAX_WEIGHT_LBS = 48000


# Get the filename and file bytes from the ipywidgets 8 upload sequence.
def extract_upload(value):
    if not value or len(value) != 1:
        raise ValueError("Attach exactly one Excel file.")
    uploaded = value[0]
    name = uploaded['name']
    # Reject unsupported or empty files before trying to read the workbook.
    if not str(name).lower().endswith('.xlsx'):
        raise ValueError("Attach an .xlsx file.")
    content = bytes(uploaded['content'])
    if not content:
        raise ValueError("The attached file is empty.")
    return str(name), content


# Output width and weight column names
def measurement_columns(frame):
    if not frame.columns.is_unique:
        raise ValueError("The worksheet has duplicate column names.")
    # Ignore differences such as capitalization, spaces, and parentheses.
    width_aliases = {'width', 'widthin', 'widthinch', 'widthinches'}
    weight_aliases = {'weightlbs', 'weightlb', 'weightpounds'}
    normalize = lambda name: re.sub(r'[^a-z0-9]', '', str(name).lower())
    widths = [column for column in frame.columns if normalize(column) in width_aliases]
    weights = [column for column in frame.columns if normalize(column) in weight_aliases]
    # Stop if a required column is missing or more than one column matches.
    if len(widths) != 1 or len(weights) != 1:
        raise ValueError("Expected one Width column and one Weight lbs column; check for missing or ambiguous headers.")
    return widths[0], weights[0]


# Convert numeric text to numbers
def numeric_measurements(series):
    values = series.map(lambda value: None if isinstance(value, bool) else value)
    return pd.to_numeric(values, errors='coerce')


# Split auction list into eligible and ineligible columns
def filter_coils(frame):
    # Convert width and weight columns into numbers
    width_column, weight_column = measurement_columns(frame)
    width = numeric_measurements(frame[width_column])
    weight = numeric_measurements(frame[weight_column])

    # Filter out
    eligible = ((width > 0) & (width <= MAX_WIDTH_INCHES)
                & (weight > 0) & (weight <= MAX_WEIGHT_LBS)).fillna(False)

    # Output eligible and ineligible
    return frame.loc[eligible].copy(), frame.loc[~eligible].copy()


# Read the worksheet
def read_bid_list(content):
    try:
        sheet = pd.read_excel(io.BytesIO(content), sheet_name=0, header=None, engine='openpyxl')
    # Turn dependency and workbook errors into messages available to the callback.
    except ImportError as error:
        raise ValueError("Excel support requires openpyxl in this notebook's Python environment.") from error
    except Exception as error:
        raise ValueError("Unable to read this Excel workbook. Attach a valid, unencrypted .xlsx file.") from error
    if sheet.empty:
        raise ValueError("The first worksheet is empty.")
    # Use the first row as column names and the remaining rows as the data.
    frame = sheet.iloc[1:].copy().reset_index(drop=True)
    frame.columns = sheet.iloc[0].tolist()
    return frame


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Close the previous interface when rerunning this cell.
for widget_name in ('upload', 'output', 'bid_list_ui'):
    previous_widget = globals().get(widget_name)
    if previous_widget is not None:
        previous_widget.close()

# Create a single-file upload button and a separate area for its confirmation.
upload = widgets.FileUpload(accept='.xlsx', multiple=False)
output = widgets.Output()
# Keep results accessible to later notebook cells; nothing is loaded initially.
df = available_coils = filtered_out_coils = None
file_name = upload_error = None


# Automatically read and filter a new attachment
def process_upload(change=None):
    # Update notebook-level variables so later cells can use the latest results.
    global df, available_coils, filtered_out_coils, file_name, upload_error
    # Clear old data first so a failed or removed upload cannot leave stale results.
    df = available_coils = filtered_out_coils = None
    file_name = upload_error = None
    # Keep the output blank unless the attachment is processed successfully.
    with output:
        clear_output()
        if not upload.value:
            return
        # Prepare all three tables before publishing any results to the notebook.
        try:
            name, content = extract_upload(upload.value)
            candidate = read_bid_list(content)
            accepted, rejected = filter_coils(candidate)
        except (ValueError, KeyError, TypeError) as error:
            # Save errors for inspection without adding messages to the widget output.
            upload_error = str(error)
            return
        # Publish the results, then print only the requested attachment confirmation.
        df, available_coils, filtered_out_coils = candidate, accepted, rejected
        file_name = name
        print(f'{file_name} attached.')


# Register automatic processing, arrange the controls, and show the interface.
upload.observe(process_upload, names='value')
bid_list_ui = widgets.VBox([upload, output])
display(bid_list_ui)
# Initialize the blank output and empty results when this cell first runs.
process_upload()


None

In [39]:
display(available_coils)

display(filtered_out_coils)

,Bid(s),Product Group,Material,Batch,Gauge,Width,Weight lbs,PIW,Translated Grade,Comment,...,Foreign Mat. Index,Defect code text 2,Defect code text 3,Defect code text 4,Unrestricted,Weight Kg,Th,Width Of Pieces,Length Of Pieces,Density
0,NaN,AL,AL-SEC,4404062540,0.023,44.895,22421,499.409734,CS Type B,NaN,...,T,1602 Missing Identity,9962 SPM CROP,NaN,10170,10170,0.593,1140.330,1960,0.0077
1,NaN,AL,AL-SEC,5302055850,0.024,48.109,16931,351.929992,CS Type B,NaN,...,P,303 Weight Under,NaN,NaN,7680,7680,0.613,1221.980,1343,0.0076
2,NaN,AL,AL-SEC,5302062370,0.024,48.113,44621,927.420863,CS Type B,NaN,...,T,808 Dents,NaN,NaN,20240,20240,0.613,1222.070,3539,0.0076
3,NaN,AL,AL-SEC,4404062550,0.029,48.269,21869,453.065114,DDS,NaN,...,4,1108 Furnace Pressure,NaN,NaN,9920,9920,0.730,1226.030,1444,0.0077
4,NaN,AL,AL-SEC,4404062480,0.029,51.791,22751,439.284818,CS Type B,NaN,...,L,NaN,NaN,NaN,10320,10320,0.726,1315.480,1406,0.0077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
846,NaN,HRP,HRP-SEC,2104830950,0.247,60.524,45128,745.621572,Structural Steel 36 Type 2,NaN,...,P,1917 Scale Pitting,822 Pits,822 Pits,20470,20470,6.270,1537.310,271,0.0078
847,NaN,HRP,HRP-SEC,2104830960,0.247,60.589,45481,750.647807,Structural Steel 36 Type 2,NaN,...,P,822 Pits,1917 Scale Pitting,822 Pits,20630,20630,6.272,1538.950,273,0.0078
848,NaN,HRP,HRP-SEC,2504389320,0.379,60.63,45388,748.606301,HSLA 80,NaN,...,T,509 Telescope ID,828 Scratch,NaN,20588,20588,9.627,1540.000,180,0.0077
850,NaN,HRP,HRP-SEC,2504404760,0.621,60.63,45633,752.647204,HSLA 50,NaN,...,G,1804 Lamination,NaN,NaN,20699,20699,15.773,1540.000,110,0.0077


,Bid(s),Product Group,Material,Batch,Gauge,Width,Weight lbs,PIW,Translated Grade,Comment,...,Foreign Mat. Index,Defect code text 2,Defect code text 3,Defect code text 4,Unrestricted,Weight Kg,Th,Width Of Pieces,Length Of Pieces,Density
9,NaN,AL,AL-SEC,4404097940,0.039,63.803,22465,352.099431,HSLA 50,NaN,...,T,979 Coating Weight High,986 Jet Lines,NaN,10190,10190,0.988,1620.590,827,0.0077
10,NaN,AL,AL-SEC,4404033480,0.043,62.022,45481,733.304311,HSLA 50,NaN,...,L,1003 Mechanical Props - Wait F,1001 Mechanical Props - Failed,104 Gauge Under,20630,20630,1.087,1575.350,1616,0.0075
11,NaN,AL,AL-SEC,5302073760,0.048,63.078,14638,232.061892,HSLA 50,NaN,...,T,992 Uncoated Spots,NaN,NaN,6640,6640,1.214,1602.190,447,0.0076
18,NaN,AL,AL-SEC,4404071770,0.056,64.655,22200,343.360916,HSLA 50,NaN,...,T,980 Coating Weight Low,NaN,NaN,10070,10070,1.415,1642.230,563,0.0077
25,NaN,AL,AL-SEC,4404092490,0.062,65.202,59105,906.490598,HSLA70,NaN,...,L,9902 Hold Pending Outcome (Con,104 Gauge Under,NaN,26810,26810,1.587,1656.140,1326,0.0077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,NaN,HRP,HRP-SEC,2104913300,0.231,60.485,57628,952.765148,HSLA 50,NaN,...,G,1804 Lamination,NaN,NaN,26140,26140,5.856,1536.330,371,0.0078
840,NaN,HRP,HRP-SEC,2104910250,0.233,60.371,58179,963.691176,HSLA 50,NaN,...,G,822 Pits,828 Scratch,103 Gauge Spike,26390,26390,5.908,1533.420,372,0.0078
841,NaN,HRP,HRP-SEC,2104910070,0.233,60.547,59987,990.750987,HSLA 50,NaN,...,G,822 Pits,1804 Lamination,103 Gauge Spike,27210,27210,5.913,1537.900,382,0.0078
844,NaN,HRP,HRP-SEC,2104913320,0.245,60.672,48236,795.029008,CS Type B,NaN,...,G,1804 Lamination,302 Weight Over,NaN,21880,21880,6.221,1541.080,291,0.0078
